In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024
):
    params = {
        "model": model,
        "max_tokens": 10000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget
        }
    else:
        params["temperature"] = 1.0

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
messages = []

task = "I have 3 apples. I ate one, then bought 2 more, and then gave half to the other. How many apples do I have left? Explain step by step."

add_user_message(messages, task)

response = chat(
    messages,
    thinking=True,
    thinking_budget=2048
)

for block in response.content:
    if block.type == "thinking":
        print("💭 CLAUDE IS THINKING")
        print(block.thinking)
        print("-" * 40)

    elif block.type == "text":
        print("✅ FINAL ANSWER")
        print(block.text)

💭 CLAUDE IS THINKING
Let me work through this step by step:

1. Starting point: 3 apples
2. Ate one: 3 - 1 = 2 apples
3. Bought 2 more: 2 + 2 = 4 apples
4. Gave half to the other: 4 ÷ 2 = 2 apples given away, so 4 - 2 = 2 apples remaining

So the answer is 2 apples.
----------------------------------------
✅ FINAL ANSWER
# Step-by-step solution:

**Starting amount:** 3 apples

**Step 1 - Ate one:**
- 3 - 1 = **2 apples**

**Step 2 - Bought 2 more:**
- 2 + 2 = **4 apples**

**Step 3 - Gave half to the other:**
- Half of 4 = 2 apples given away
- 4 - 2 = **2 apples**

**Final answer: You have 2 apples left.**
